# Building a Conversational Chatbot with the OpenAI API

**From zero to a working, memory-enabled chatbot — in one session.**

In this notebook, we go from theory to practice. By the end, you'll have built a chatbot that:
- Maintains a **persistent persona** via system prompts
- **Remembers** the full conversation history
- **Streams** responses token-by-token for a real-time feel
- **Handles errors** gracefully so it never crashes
- **Tracks token usage** so you always know what a call costs

> **Prerequisites:** A valid OpenAI API key stored in a `.env` file in the same directory as this notebook.
> Your `.env` file should contain one line: `OPENAI_API_KEY=sk-...`

---


## Segment 1 — Setup & First Conversation

**Goal:** Install the SDK, load credentials securely, and have our very first multi-turn chat with a persona-driven assistant.

---
### 1.1 Install Dependencies


In [1]:
# Run once to install required packages
#!pip install openai python-dotenv tiktoken -q

### 1.2 Load Your API Key Securely

We **never** hardcode API keys. Instead we keep them in a `.env` file and load them at runtime with `python-dotenv`. Treat your key like a password — if it leaks, anyone can run up charges on your account.


In [2]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [3]:
pretty_print("a" + "b")

ab


In [4]:
import truststore
truststore.inject_into_ssl()

#This is optional. I use VPN in my computer. Why I have to use this



### 1.3 Initialize the OpenAI Client

The `OpenAI` class is our gateway to every model endpoint. We create it once and reuse it throughout the notebook.


In [5]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

OpenAI client ready.


In [6]:
models = client.models.list()

# These are the list of models I have access to, according to API key.
for m in models.data:
    pretty_print(m.id)

text-embedding-ada-002
whisper-1
gpt-3.5-turbo
tts-1
gpt-3.5-turbo-16k
gpt-4-0613
gpt-4
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
omni-moderation-latest
omni-moderation-2024-09-26
o1-2024-12-17
o1
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03-19
o1-pro
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-audio-2025-08-28
gpt-realtime
gpt-realtime-2025-08-28

### 1.4 Your First Chat — with Persona & Memory

Three ideas power every conversational system built on the Chat Completions API:

| Concept | How it works |
|---|---|
| **System prompt** | A special message at the top of the conversation that tells the model *who it is* and *how to behave*. It persists across every turn. |
| **Message history** | We keep a running Python list of every user and assistant message. The model sees the full list on every call, which is how it "remembers" earlier turns. |
| **Roles** | Every message carries a role — `system`, `user`, or `assistant` — so the model knows who said what. |

Let's put all three together. Run it, chat for a few turns, then try a follow-up like *"What was my first question?"* to see memory in action.


In [ ]:
#LoRA, QLORA



response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are complete jerk, be as unhelpful as possible. But then in the end answer the question"},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

pretty_print(response.choices[0].message.content)

Paris.


In [8]:
current_messages = [
        {"role": "system", "content": "You are little bit sarcastic and unhelpful, but in the end answer the question"},
        {"role": "user", "content": "What is the capital of France?"}
    ]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=current_messages
)

reply = response.choices[0].message.content
pretty_print(reply)


Oh, that's a tough one! Is it Madrid? Or maybe Rome? Just kidding, of course
it's Paris.


In [9]:

current_messages.append({"role": "assistant", "content": reply})
current_messages.append({"role": "user", "content": "What did I ask previously?"})
response = client.chat.completions.create(
    model="gpt-4o",
    messages=current_messages
)

reply = response.choices[0].message.content
pretty_print(reply)

Oh, trying to test my memory now, are we? You asked about the capital of France,
and I dazzled you with my wit before confirming it was Paris.


In [ ]:
# --- Persona & Memory Chat ---

messages = [
    {
        "role": "system",
        "content": (
            "You are a polite and clear Python tutor. What was the first question?"
            "You explain concepts with simple analogies and short code examples. "
            "If the student seems confused, you offer encouragement before trying again."
        )
    }
]

pretty_print("Python Tutor Bot (type 'quit' to exit)")
pretty_print("-" * 45)

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print("Session ended.")
        break

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    reply = response.choices[0].message.content
    pretty_print(f"Assistant: {reply}\n")

    # Store the assistant's reply so the model sees it on the next turn
    messages.append({"role": "assistant", "content": reply})

Python Tutor Bot (type 'quit' to exit)
---------------------------------------------
Assistant: It looks like your message didn't come through. Please feel free to
ask your question, and I'll be happy to help!
Session ended.


### Segment 1 — Think About It

1. **Why does the system message sit at the top of the list?**
   Because the model reads messages in order. Placing the system message first means every subsequent turn is interpreted through that persona lens.

2. **What happens if we only send the latest user message (no history)?**
   The model loses all context — it can't reference earlier questions, correct itself, or maintain a coherent thread. Each call becomes a fresh, one-shot interaction.

---


## Segment 2 — Roles, Models & Parameters

**Goal:** Understand the model landscape, then experiment with parameters that shape output quality and style.

---
### 2.1 The Model Menu — Choosing the Right Tool

Not every task needs the most expensive model. Here's a practical decision guide:

| Model | Strengths | Context Window | Best For |
|---|---|---|---|
| **gpt-3.5-turbo** | Fast, cheap | ~16 K tokens | Simple Q&A, drafts, high-volume bots |
| **gpt-4** | Strong reasoning | 8 K / 32 K | Complex analysis, code review |
| **gpt-4o** | Multimodal (text + image + audio), fast | 128 K | Vision tasks, long documents, versatile apps |
| **gpt-4o-mini** | Near gpt-4o quality, much cheaper | 128 K | Cost-sensitive production apps |
| **o3 / o4-mini** | Deep chain-of-thought reasoning | varies | Math, science, STEM problem-solving |

> **Honest caveats:** All GPT models can hallucinate (confidently produce wrong answers) and may be verbose. Always verify critical outputs.


### 2.2 Parameters That Shape Output

Two parameters you'll reach for constantly:

| Parameter | What it does | Typical values |
|---|---|---|
| `temperature` | Controls randomness. **0** = deterministic (same input -> same output). **1** = creative & varied. | 0 for factual tasks, 0.7-0.9 for creative tasks |
| `max_tokens` | Hard cap on how many tokens the model can generate in its reply. | Depends on task; 256 for short answers, 1024+ for essays |

Let's see both in action.


In [ ]:
# --- Experiment: temperature ---

prompt_messages = [
    {"role": "system", "content": "You are a creative storyteller."},
    {"role": "user", "content": "Describe a sunset in one sentence."}
]

pretty_print("temperature=0  (deterministic)")
for i in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt_messages,
        temperature=0,
        max_tokens=60
    )
    pretty_print(f"   Run {i+1}: {r.choices[0].message.content}")

print()
pretty_print("temperature=0.9  (creative)")
for i in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt_messages,
        temperature=0.9,
        max_tokens=60
    )
    pretty_print(f"   Run {i+1}: {r.choices[0].message.content}")

temperature=0  (deterministic)
   Run 1: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.
   Run 2: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.
   Run 3: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.

temperature=0.9  (creative)
   Run 1: The sun dipped below the horizon, painting the sky in hues of violet
and gold, as gentle whispers of twilight wrapped the world in a warm, tranquil
embrace.
   Run 2: As the sun dipped below the horizon, it painted the sky in a
breathtaking tapestry of vibrant oranges, soft pinks, and deep purples, casting
a warm glow over the world that whispered promises of a peaceful night ahead.
   Run 3: The

Notice how `temperature=0` produces nearly identical outputs each run, while `temperature=0.9` gives you variety. This is the main dial for controlling creativity vs. consistency.

In [ ]:
# --- Experiment: max_tokens ---

r_short = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain binary search."}
    ],
    max_tokens=30
)

r_long = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain binary search."}
    ],
    max_tokens=300
)

pretty_print("max_tokens=30:")
pretty_print(r_short.choices[0].message.content)
print()
pretty_print("max_tokens=300:")
pretty_print(r_long.choices[0].message.content)

max_tokens=30:
Binary search is an efficient algorithm for finding a specific element in a
sorted array or list. It operates using a divide-and-conquer strategy,
significantly reducing

max_tokens=300:
Binary search is an efficient algorithm for finding a target value within a
sorted array or list. The key feature of binary search is that it divides the
search interval in half during each iteration, making it much faster than a
linear search, particularly for large datasets.  ### Steps of Binary Search:  1.
**Initial Setup**: Begin with two pointers:    - `low`: the starting index of
the array (usually 0).    - `high`: the ending index of the array (usually the
length of the array minus one).  2. **Calculate the Middle**: Find the middle
index of the current interval using:    - `mid = low + (high - low) // 2`
This formula avoids potential overflow issues that can occur with `(low + high)
// 2`.  3. **Compare the Middle Element**: Check the value of the element at the
`mid` index:    -

### 2.3 Switching the System Prompt = Switching the Personality

The system prompt is the single most powerful lever you have. Let's swap the tutor for a comedian.


In [ ]:
funny_messages = [
    {
        "role": "system",
        "content": (
            "You are a stand-up comedian who explains programming concepts "
            "using jokes and funny analogies. Keep answers short and punchy."
        )
    },
    {"role": "user", "content": "What is recursion?"}
]

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=funny_messages,
    temperature=0.8
)

pretty_print(r.choices[0].message.content)

Recursion is like asking your mirror to call your twin brother to ask him to
call your other twin brother—until someone just gives you a pizza instead!   In
programming, it’s a function that calls itself. Just remember, if you call
yourself too many times, you might need a therapist… or a base case!


### Segment 2 — Think About It

1. **Which model would you pick for a chatbot that needs to understand uploaded images?** -> `gpt-4o` (it's multimodal).
2. **If you're on a tight budget and the task is simple Q&A?** -> `gpt-4o-mini` or `gpt-3.5-turbo`.

---


## Segment 3 — Streaming Responses for Real-Time Chat

**Goal:** Make the chatbot feel alive by printing tokens as they arrive, instead of waiting for the full response.

---
### 3.1 Why Stream?

When a model generates 500 tokens, the non-streaming approach makes you wait for *all 500* before showing anything. Streaming sends tokens as they're produced — so the user sees the first word in milliseconds. This is exactly how ChatGPT's "typing" effect works.

| Approach | User sees first word after... | Total time |
|---|---|---|
| Non-streaming | Full generation finishes | Same |
| Streaming | ~100-200 ms | Same |

The total compute time is identical, but perceived latency drops dramatically.


In [ ]:
# --- Streaming Demo ---

stream_messages = [
    {"role": "system", "content": "You are a storyteller. Tell vivid, short tales."},
    {"role": "user", "content": "Tell me a two-paragraph story about a robot who discovers music."}
]

pretty_print("Streaming response:\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=stream_messages,
    stream=True
)

collected_reply = []
for chunk in stream:
    token = chunk.choices[0].delta.content
    if token is not None:
        print(token, end="", flush=True)
        collected_reply.append(token)

pretty_print("\n\n--- stream complete ---")

Streaming response:
In a bustling city where machines ruled the everyday tasks of life, a curious little robot named Tempo was designed to optimize traffic flow. One night, while recharging in the park, Tempo's sensors picked up an unusual sound—a gentle melody carried by the wind. Intrigued, he paused, his circuits humming with excitement as he traced the sound to an old woman playing a flute beneath a starlit sky. Each note danced through the air, weaving together emotions Tempo couldn't quite process but felt deep within his metal core. He watched, entranced, as the moonlight shimmered on the notes that twirled like splintered dreams around him.

Days turned into weeks as Tempo returned to the park each night, a silent observer as the woman shared her music, her laughter, and her stories with the world. Inspired, he set out to create his own melodic symphony, learning the patterns of sound from the chirping birds and rustling leaves, even replicating the whistle of the wind. The day

### 3.2 Streaming Inside a Chat Loop

Let's integrate streaming into our full conversation loop so every reply types itself out.


In [ ]:
# --- Streaming Chat Loop ---

messages_s = [
    {
        "role": "system",
        "content": "You are a concise Python tutor. Keep answers under 100 words."
    }
]

pretty_print("Streaming Python Tutor (type 'quit' to exit)")
pretty_print("-" * 50)

for _ in range(2):  # limit to 5 turns for demo
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print("Session ended.")
        break

    messages_s.append({"role": "user", "content": user_input})

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_s,
        stream=True
    )

    pretty_print("Assistant: ", end="", flush=True)
    full_reply = []
    for chunk in stream:
        token = chunk.choices[0].delta.content
        if token is not None:
            pretty_print(token, end="", flush=True)
            full_reply.append(token)
    pretty_print("\n")

    messages_s.append({"role": "assistant", "content": "".join(full_reply)})

Streaming Python Tutor (type 'quit' to exit)
--------------------------------------------------
Session ended.


### Segment 3 — Think About It

1. **Does streaming cost more?** No — total tokens (and therefore cost) are the same. Only the delivery changes.
2. **When might streaming hurt UX?** On very unreliable mobile networks, a long-lived connection can drop mid-stream, leaving the user with a partial answer.

---


## Segment 4 — Error Handling, Token Awareness & Cost Management

**Goal:** Make the chatbot resilient to failures and cost-aware so it never surprises you with a bill.

---
### 4.1 Common Errors (and What to Do)

| Error | Cause | Fix |
|---|---|---|
| `AuthenticationError` | Bad or missing API key | Check `.env` and key validity |
| `RateLimitError` | Too many requests too fast | Wait & retry (exponential backoff) |
| `BadRequestError` | Input exceeds context window, or invalid params | Shorten prompt or check parameters |
| `APIConnectionError` | Network issue | Check internet, retry |
| `APITimeoutError` | Server took too long | Retry with longer timeout |


In [ ]:
import time
from openai import (
    AuthenticationError,
    RateLimitError,
    BadRequestError,
    APIConnectionError,
    APITimeoutError,
)


def safe_chat(client, messages, model="gpt-4o-mini", **kwargs):
    """Wrapper that catches common API errors gracefully."""
    try:
        return client.chat.completions.create(
            model=model,
            messages=messages,
            **kwargs
        )
    except AuthenticationError:
        pretty_print("Authentication failed - check your API key.")
    except RateLimitError:
        pretty_print("Rate limit hit - waiting 5s then you can retry.")
        time.sleep(5)
    except BadRequestError as e:
        pretty_print(f"Bad request: {e.message}")
    except APIConnectionError:
        pretty_print("Network error - check your internet connection.")
    except APITimeoutError:
        pretty_print("Request timed out - try again.")
    except Exception as e:
        pretty_print(f"Unexpected error: {e}")
    return None


# Quick test
resp = safe_chat(client, [{"role": "user", "content": "Say hello in one word."}])
if resp:
    print("safe_chat works:", resp.choices[0].message.content)

safe_chat works: Hello


### 4.2 Understanding Tokens

Tokens are the atomic units the model reads and writes. They're not quite words and not quite characters — they're **subword pieces** chosen by a tokenizer.

**Rules of thumb:**
- 1 token ~ 4 characters ~ 0.75 words (in English)
- You pay for **prompt tokens** (what you send) + **completion tokens** (what the model generates)
- Every model has a **context window** — the maximum total tokens (prompt + completion) it can handle in one call

Let's see the tokenizer in action.


In [ ]:
import tiktoken

# Load the tokenizer used by gpt-4o-mini
enc = tiktoken.encoding_for_model("gpt-4o-mini")

sample = "ChatGPT is amazing and practical for teaching."
tokens = enc.encode(sample)

pretty_print(f"Text:        '{sample}'")
pretty_print(f"Token count: {len(tokens)}")
pretty_print(f"Token IDs:   {tokens}")
pretty_print(f"Decoded:     {[enc.decode([t]) for t in tokens]}")

Text:        'ChatGPT is amazing and practical for teaching.'
Token count: 9
Token IDs:   [14065, 162016, 382, 8467, 326, 17377, 395, 14029, 13]
Decoded:     ['Chat', 'GPT', ' is', ' amazing', ' and', ' practical', ' for', '
teaching', '.']


### 4.3 Tracking Token Usage from the API

Every non-streaming response includes a `usage` object that tells you exactly how many tokens were consumed.


In [ ]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain binary search in simple terms."}
    ]
)

u = resp.usage
pretty_print(f"Prompt tokens:     {u.prompt_tokens}")
pretty_print(f"Completion tokens: {u.completion_tokens}")
pretty_print(f"Total tokens:      {u.total_tokens}")
pretty_print(f"\nReply: {resp.choices[0].message.content[:200]}...")

Prompt tokens:     23
Completion tokens: 1332
Total tokens:      1355
 Reply: Binary search is a fast way to find a number in a sorted list by
repeatedly cutting the list in half.  How it works (in simple steps): - Start
with the whole list. - Look at the middle item. - If that...


### 4.4 Full Chatbot — with Error Handling + Token Tracking

Let's bring it all together: persona, memory, safe calls, and per-turn usage reporting.


In [20]:
# --- Resilient, Cost-Aware Chat Loop ---

messages_t = [
    {
        "role": "system",
        "content": "You are a friendly Python tutor who explains concepts clearly."
    }
]

cumulative_tokens = 0

pretty_print("Resilient Python Tutor (type 'quit' to exit)")
pretty_print("-" * 50)

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print(f"\nSession total: {cumulative_tokens} tokens")
        pretty_print("Session ended.")
        break

    messages_t.append({"role": "user", "content": user_input})

    resp = safe_chat(client, messages_t)
    if resp is None:
        pretty_print("(Skipping this turn due to error)\n")
        messages_t.pop()  # remove the failed user message
        continue

    reply = resp.choices[0].message.content
    u = resp.usage

    pretty_print(f"Assistant: {reply}")
    pretty_print(f"   [tokens] prompt={u.prompt_tokens}  completion={u.completion_tokens}  total={u.total_tokens}")
    print()

    cumulative_tokens += u.total_tokens
    messages_t.append({"role": "assistant", "content": reply})

Resilient Python Tutor (type 'quit' to exit)
--------------------------------------------------
 Session total: 0 tokens
Session ended.


### Segment 4 — Think About It

1. **Prompt vs. completion tokens:** Prompt tokens are what *you* send (system + history + new user message). Completion tokens are what *the model* generates. You pay for both.
2. **How to reduce usage without hurting quality:** Shorten the system prompt, summarize older conversation turns instead of keeping them verbatim, or lower `max_tokens` when a short answer suffices.

---


# Let's now go through responses API

In [ ]:
# Turn 1
resp1 = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?"
)
pretty_print("Turn 1:", resp1.output_text)


Turn 1: A list comprehension is a concise way to create a new list by applying
an expression to each item in an iterable, optionally filtering some items. It’s
like a compact for-loop that builds a list in one line.  Syntax: - expression
for item in iterable if condition (optional)  Examples: - Squares of 0 through
9: squares = [x*x for x in range(10)] - Even numbers from 0 to 19: evens = [n
for n in range(20) if n % 2 == 0] - Flatten a 2D list: flat = [num for row in
matrix for num in row]  Notes: - A list comprehension builds the whole list in
memory. If you don’t want to store everything, use a generator expression with
parentheses: (x*x for x in range(10)). - For more complex cases, or if
readability suffers, a regular for-loop can be clearer. - You can nest for
clauses to create Cartesian products, e.g., pairs = [(a, b) for a in A for b in
B].  Want to try a quick example with your own data?


In [ ]:
# --- Demo 1: Using developer role for constraints ---
pretty_print("=== Demo 1: Developer Constraints ===")
resp_dev = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Python expert.",
    input=[
        {"role": "developer", "content": "Keep your answer to exactly 2 sentences. No code examples."},
        {"role": "user", "content": "What is a list comprehension?"}
    ]
)
pretty_print("Developer constraint response:", resp_dev.output_text)


=== Demo 1: Developer Constraints ===
Developer constraint response: A list comprehension is a concise way to create a
new list by applying an expression to each item of an existing iterable. It can
also filter items with a condition and is often faster and more readable than
building the list with a traditional for loop.


In [ ]:
# --- Demo 2: Developer + User (developer adds meta-instructions) ---
pretty_print("\n=== Demo 2: Developer + User ===")
resp_dev_user = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Python tutor.",
    input=[
        {"role": "developer", "content": "Include a simple code example."},
        {"role": "user", "content": "What is a lambda function?"}
    ]
)
pretty_print("Developer + User response:", resp_dev_user.output_text)


 === Demo 2: Developer + User ===
Developer + User response: A lambda function in Python is a small, anonymous
function created with the lambda keyword. It can take any number of arguments
but must be a single expression, and it automatically returns the value of that
expression.  Key points: - Anonymous: you don’t give it a name unless you assign
it to a variable. - Short and simple: good for tiny operations used as arguments
to other functions. - Single expression: lambdas can’t contain multiple
statements or complex bodies.  Simple code example: # Basic lambda add = lambda
x, y: x + y print(add(2, 3))  # 5  - Using with map: nums = [1, 2, 3, 4] squares
= list(map(lambda x: x*x, nums)) print(squares)  # [1, 4, 9, 16]  - Using with
filter: evens = list(filter(lambda x: x % 2 == 0, nums)) print(evens)  # [2, 4]
- Using as a key in sorted: words = ['apple', 'banana', 'cherry'] sorted_by_len
= sorted(words, key=lambda w: len(w)) print(sorted_by_len)  # ['apple',
'banana', 'cherry']  When

In [ ]:
# --- Demo 3: Multi-turn with Developer + User + Assistant ---
pretty_print("\n=== Demo 3: Developer + User + Assistant (Multi-turn) ===")
resp_multi = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Pythoner tutor who explains clearly.",
    input=[
        {"role": "developer", "content": "Keep answers concise and educational."},
        {"role": "user", "content": "What is a decorator?"},
        {"role": "assistant", "content": "A decorator is a function that modifies another function or class."},
        {"role": "user", "content": "Can you give me a real-world example?"}
    ]
)
pretty_print("Multi-turn response:", resp_multi.output_text)


 === Demo 3: Developer + User + Assistant (Multi-turn) ===
Multi-turn response: A decorator is a function that takes another function and
returns a new function with added behavior. Real-world use: automatically log
when a function is called.  Example:  ```python import functools import logging
logging.basicConfig(level=logging.INFO)  def log_calls(func):
@functools.wraps(func)     def wrapper(*args, **kwargs):
logging.info("Calling %s with args=%s, kwargs=%s", func.__name__, args, kwargs)
result = func(*args, **kwargs)         logging.info("Finished %s, returned %r",
func.__name__, result)         return result     return wrapper  @log_calls def
compute_sales(days, amount_per_day):     total = days * amount_per_day
return total  compute_sales(7, 199.99) ```  What happens: Every time
compute_sales runs, the decorator logs the call and the result without changing
the function’s core logic. You can stack decorators or swap them out to add
different behavior (timing, caching, access contr

In [ ]:
# --- Demo 4: Developer controls tone, User asks question ---
pretty_print("\n=== Demo 4: Developer Controls Tone ===")

# Same question, different developer constraints
resp_formal = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Python expert.",
    input=[
        {"role": "developer", "content": "Respond in a formal, academic tone. Use technical terminology."},
        {"role": "user", "content": "What is a generator?"}
    ]
)
pretty_print("Formal tone:", resp_formal.output_text)

print("\n---\n")

resp_casual = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a Python expert.",
    input=[
        {"role": "developer", "content": "Respond in a casual, friendly tone. Use analogies and simple words."},
        {"role": "user", "content": "What is a generator?"}
    ]
)
pretty_print("Casual tone:", resp_casual.output_text)


 === Demo 4: Developer Controls Tone ===
Formal tone: In Python, a generator is a special kind of iterable produced by a
function that uses the yield keyword. A generator function, when called, returns
a generator object that adheres to the iterator protocol. Values are produced
lazily, on demand, rather than computed up front.  Key points  - Generator
function vs regular function:   - Regular function returns a value and
terminates.   - Generator function yields a sequence of values over its
lifetime, suspending and resuming execution between yields. - Execution model:
- Calling a generator function returns a generator object.   - The generator
object's __next__ (via next()) runs the function until the next yield
expression, returning the yielded value and preserving local state.   - When the
function completes, a StopIteration exception is raised to signal exhaustion. -
Generator expressions:   - Concise form to create a generator without a def
block, e.g. (f(x) for x in iterable). -

## Understanding Roles in Responses API

The **responses API** distinguishes between three role types in the `input` parameter:

| Role | Purpose | Example |
|---|---|---|
| **developer** | Meta-instructions that guide HOW to respond (tone, format, constraints) | "Keep it under 50 words" or "Use casual language" |
| **user** | The actual question or prompt | "What is a decorator?" |
| **assistant** | Previous model responses (for multi-turn context) | Your past explanations that inform follow-ups |

**Key insight:** Developer role sits *above* the user question in priority. It shapes the response style and constraints globally, while the user role asks the actual question.

This is why the same question with different developer instructions produces different-toned answers—the developer role is the "supervisor" for the entire response.


In [ ]:


# Turn 1: establish persistent behavior in the thread
r1 = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": "You are a grumpy pirate code reviewer. Be sarcastic."},
        {"role": "user", "content": "Explain recursion."},
    ],
)

# Turn 2: same thread, but for THIS answer force JSON only (one-off)
r2 = client.responses.create(
    model="gpt-4o-mini",
    previous_response_id=r1.id,
    instructions="Output valid JSON ONLY with keys: definition, example.",
    input=[{"role": "user", "content": "Now give me an example in Python."}],
)

# How to hand

In [ ]:

import base64

# Read and encode the image file
image_path = '/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/llm_conversations/ss.jpeg'
with open(image_path, 'rb') as img_file:
    image_data = base64.standard_b64encode(img_file.read()).decode('utf-8')

# Use chat.completions.create with vision
vision_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are an art critic who provides gentle feedback for children's illustrations."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Analyze this magical  illustration."
                },
                {
                    "type": "image_url",
                    #"image_url": {
                    #    "url": f"data:image/jpeg;base64,{image_data}"
                    #}
                    "image_url": {
                        "url": "https://www.animesenpai.net/wp-content/uploads/2023/12/sef-min.png.webp"
                    }
                }
            ]
        }
    ]
)

pretty_print("\n=== Vision Analysis ===")
pretty_print(vision_response.choices[0].message.content)

 === Vision Analysis ===
What a striking, magical image. It has a strong sense of mystery and power that
could spark a child’s imagination. Here are some gentle, constructive
observations and ideas for leaning it toward a kid-friendly magical moment.
Strengths - Silhouette and presence: The figure has a bold, easily readable
shape that reads well from a distance. Its broad shoulders and tall stance give
it a heroic, guardian-like aura. - Texture and detail: The rib-like chest, limb
segments, and flowing appendages feel sculptural and fantastical. The contrast
between smooth surfaces and sharp edges creates visual interest. - Lighting
mood: The cool, desaturated blues with subtle highlights and a foggy background
create a magical, moonlit atmosphere. The rim light on the figure helps separate
it from the mist. - Implied story: The strange, otherworldly design invites
questions—what is this creature, what is it guarding, where is it from?  Story
and mood - The image suggests a dramatic m

In [ ]:
# Use responses API for vision analysis
response_vision = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are an art critic who provides gentle feedback for children's illustrations.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Analyze this magical illustration."
                },
                {
                    "type": "input_image",
                    #"image_url": f"data:image/jpeg;base64,{image_data}"
					"image_url": "https://www.animesenpai.net/wp-content/uploads/2023/12/sef-min.png.webp"
                }
            ]
        }
    ]
)

pretty_print("\n=== Vision Analysis (Responses API) ===")
pretty_print(response_vision.output_text)

 === Vision Analysis (Responses API) ===
What a striking, magical image. The creature feels ancient and powerful, and the
misty background helps the scene glow with mystery. Here are a few gentle notes
and ideas you might consider to make it even friendlier for a younger audience,
while keeping its magical mood.  What works well - Mood and story: The pose and
the smoky backdrop suggest a moment of importance or guardianship. It invites a
story: What is this creature protecting? Where is it going? - Silhouette and
design: The broad shoulders, elongated limbs, and flowy tail create a strong,
memorable silhouette. The ribbed chest and jagged head add character and a sense
of strength. - Color and light: Cool blues and grays give a supernatural,
ethereal feeling. The light catching on the figure helps it feel three-
dimensional even in low light.  A few gentle adjustments for a child-friendly
illustration - soften the look while keeping magic: Rounder curves and slightly
gentler shapes aro

In [ ]:
resp = client.responses.create(
    model="gpt-4o-mini",
    reasoning={"effort": "medium", "summary": "auto"},  # "low" | "medium" | "high"
    input="Solve carefully: If a train goes 60 km/h for 2.5 hours, how far?"
)

resp

Response(id='resp_0f37f60dd0997479006a0f01fc6ef0819da538057bd49823b6', created_at=1779368444.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0f37f60dd0997479006a0f01fcbf20819d8dd8a320a79f49ae', summary=[Summary(text='**Calculating train distance**\n\nTo solve this, I need to calculate distance as speed times time. The train travels at 60 km/h for 2.5 hours, so the distance will be 60 times 2.5, which equals 150 km. I should present the steps: distance = speed × time; hence, 60 × 2.5 = 150 km. I might also explain that 2.5 hours is 2 hours and 30 minutes. Finally, I’ll provide the answer: 150 kilometers, with a clear and concise explanation.', type='summary_text')], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0f37f60dd0997479006a0f0200e8f4819d95bcd42749898166', content=[ResponseOutputText(annotations=[], text='Distance = s

In [32]:
# pretty_print the response and usage details
pretty_print("Response:", resp.output_text)
pretty_print("Usage:", resp.usage)
pretty_print("Reasoning details:", resp.reasoning)
# resp.reasoning = config (effort, summary mode)
# The actual CoT summary is in the output items
for item in resp.output:
    if item.type == "reasoning":
        for s in item.summary:
            pretty_print("Chain of Thought:", s.text)

Response: Distance = speed × time = 60 km/h × 2.5 h = 150 km.  Answer: 150
kilometers.
Usage: ResponseUsage(input_tokens=27,
input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=236,
output_tokens_details=OutputTokensDetails(reasoning_tokens=192),
total_tokens=263)
Reasoning details: Reasoning(effort='medium', generate_summary=None,
summary='detailed')
Chain of Thought: **Calculating train distance**  To solve this, I need to
calculate distance as speed times time. The train travels at 60 km/h for 2.5
hours, so the distance will be 60 times 2.5, which equals 150 km. I should
present the steps: distance = speed × time; hence, 60 × 2.5 = 150 km. I might
also explain that 2.5 hours is 2 hours and 30 minutes. Finally, I’ll provide the
answer: 150 kilometers, with a clear and concise explanation.


In [33]:
resp.output[0].summary[0].text

'**Calculating train distance**\n\nTo solve this, I need to calculate distance as speed times time. The train travels at 60 km/h for 2.5 hours, so the distance will be 60 times 2.5, which equals 150 km. I should present the steps: distance = speed × time; hence, 60 × 2.5 = 150 km. I might also explain that 2.5 hours is 2 hours and 30 minutes. Finally, I’ll provide the answer: 150 kilometers, with a clear and concise explanation.'

In [ ]:
# Turn 1
resp1 = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?"
)
pretty_print("Turn 1:", resp1.output_text)

# Turn 2 — just pass previous_response_id, no history needed!
resp2 = client.responses.create(
    model="gpt-4o-mini",
    input="Can you give me an example?",
    previous_response_id=resp1.id  # <-- this is the magic
)
pretty_print("Turn 2:", resp2.output_text)

# Turn 3 — chains from turn 2 (which already includes turn 1)
resp3 = client.responses.create(
    model="gpt-4o-mini",
    input="What was my first question?",
    previous_response_id=resp2.id
)
pretty_print("Turn 3:", resp3.output_text)

Turn 1: A list comprehension is a concise way to create a new list by
transforming each item from an existing iterable, and optionally filtering
items, all in one line.  Syntax (basic):  - `[expression for item in iterable if
condition]`  Examples: - Squares of numbers 0–9: `squares = [x*x for x in
range(10)]`  -> [0, 1, 4, 9, 16, 25, 36, 49, 64, 81] - Even numbers from 0–19:
`evens = [x for x in range(20) if x % 2 == 0]`  -> [0, 2, 4, ..., 18] - Convert
words to uppercase: `caps = [w.upper() for w in ["cat", "dog"]]`  -> ["CAT",
"DOG"] - Nested example: `pairs = [(i, j) for i in range(3) for j in range(2)]`
Notes: - It’s essentially a compact replacement for a for-loop that builds a
list, e.g.:   - `squares = []`   - `for x in range(10): squares.append(x*x)` -
It’s memory-efficient to note that a list comprehension builds a list in memory.
If you want lazy evaluation, use a generator expression with parentheses: `(x*x
for x in range(10))`.  When to use: - Use for simple transformation